# Assignment 04: Apriori Market Basket Analysis and PCA Projection

This notebook contains two parts:

- **Assignment 4A:** Restaurant sales analysis using transaction preprocessing, custom Apriori itemset mining, and a discount-impact simulation.
- **Assignment 4B:** Numeric data analysis using interactive 2D/3D plots, centering, projections, covariance, eigen decomposition, and PCA-style projection.

The original notebook was prepared for a Data Mining coursework assignment and has been organized into a cleaner, reproducible structure for GitHub.

In [ ]:
# Optional: mount Google Drive when running this notebook in Google Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped. Running outside Colab or Google Drive is unavailable.")


In [ ]:
# Install dependencies in Colab if needed.
!pip -q install pandas numpy openpyxl mlxtend plotly kagglehub

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from mlxtend.frequent_patterns import apriori


## Part 4A: Restaurant Sales and Apriori Analysis

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
from pathlib import Path

# Update this path if the restaurant sales dataset is stored somewhere else.
candidate_paths = [
    Path("data/assignemnt4.csv"),
    Path("data/Restaurant_sales_analysis_by_dish_by_date.xlsx"),
    Path("/content/drive/MyDrive/Dataset/assignemnt4.csv"),
    Path("/content/drive/MyDrive/Dataset/Restaurant_sales_analysis_by_dish_by_date.xlsx"),
]

for data_path in candidate_paths:
    if data_path.exists():
        if data_path.suffix.lower() in [".xlsx", ".xls"]:
            df_4a = pd.read_excel(data_path)
        else:
            df_4a = pd.read_csv(data_path)
        break
else:
    raise FileNotFoundError(
        "Restaurant sales dataset not found. Place the file in data/ or update candidate_paths."
    )

print("Loaded restaurant dataset from:", data_path)
df_4a.head()


,Date,OrderId,Vegetarian's Delight,Price,BBQ Chicken Pizza,Price.1,Smoked Chicken Sandwich,Price.2,Penne Alfredo,Price.3,...,Chicken Caesar Salad,Price.111,Citrus Bliss,Price.112,Kaffeehaus Special,Price.113,Full English Breakfast,Price.114,Light English Breakfast,Price.115
0,8-Jun-2023,1995189,1,420,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,8-Jun-2023,1990493,0,0,1,790,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,8-Jun-2023,1990846,0,0,1,360,1,170,1,360,...,0,0,0,0,0,0,0,0,0,0
3,8-Jun-2023,1991366,0,0,1,360,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8-Jun-2023,1992304,0,0,1,360,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 4A.1 Data Quality Checks

In [ ]:
total_missing = df_4a.isna().sum().sum()
print("Total missing values in dataset:", total_missing)


Total missing values in dataset: 0


In [ ]:
missing_by_col = df_4a.isna().sum()
missing_by_col = missing_by_col[missing_by_col > 0]

if len(missing_by_col) == 0:
    print("No missing values found in any column.")
else:
    print("Columns with missing values:")
    display(missing_by_col.to_frame("missing_count"))


No missing values found in any column.


In [ ]:
duplicate_rows = df_4a.duplicated().sum()
print("Duplicate rows:", duplicate_rows)


Duplicate rows: 0


In [ ]:
if "OrderID" in df_4a.columns:
    print("Duplicate OrderID values:", df_4a["OrderID"].duplicated().sum())


In [ ]:
df_4a.info()


In [ ]:
df = df_4a.copy()

first_two = df.columns[:2].tolist()
print("First two columns:", first_two)

# Heuristic check (no assumptions, just flags)
c0, c1 = first_two[0].lower(), first_two[1].lower()
looks_like_date = ("date" in c0) or ("date" in c1)
looks_like_order = ("order" in c0 and "id" in c0) or ("order" in c1 and "id" in c1) or ("orderid" in c0) or ("orderid" in c1)

print("Looks like Date present in first two cols? ", looks_like_date)
print("Looks like OrderID present in first two cols?", looks_like_order)

# Optional: attempt to parse the 1st column as date (won't change your df)
try_parse = pd.to_datetime(df[first_two[0]], errors="coerce")
print(f"Parsed-as-date (col 0) non-null: {try_parse.notna().sum()} / {len(df)}")


First two columns: ['Date', 'OrderId']
Looks like Date present in first two cols?  True
Looks like OrderID present in first two cols? True
Parsed-as-date (col 0) non-null: 6307 / 6307


In [ ]:
df = df_4a.copy()

all_cols = df.columns.tolist()

dish_cols = []
price_cols = []

# After Date and OrderId, columns alternate: Dish, Price
for i in range(2, len(all_cols), 2):
    dish_cols.append(all_cols[i])
    price_cols.append(all_cols[i + 1])


print("Dish columns:", len(dish_cols))
print("Price columns:", len(price_cols))

print("\nFirst 5 dish columns:", dish_cols[:5])
print("First 5 price columns:", price_cols[:5])


Dish columns: 116
Price columns: 116

First 5 dish columns: ["Vegetarian's Delight", 'BBQ Chicken  Pizza', 'Smoked Chicken Sandwich', 'Penne Alfredo', 'Pepsi']
First 5 price columns: ['Price', 'Price.1', 'Price.2', 'Price.3', 'Price.4']


In [ ]:
# Convert dish and price columns to numeric safely
for c in dish_cols + price_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Check missing values introduced
print("NaNs in dish columns:", df[dish_cols].isna().sum().sum())
print("NaNs in price columns:", df[price_cols].isna().sum().sum())


NaNs in dish columns: 0
NaNs in price columns: 0


### 4A.2 Transaction Matrix for Apriori

Apriori uses binary transaction data. A dish value greater than zero means the dish was included in the order, so it is converted to 1. Otherwise, it is converted to 0.

In [ ]:
transactions = df[dish_cols].copy()
transactions = (transactions.fillna(0) > 0).astype(int)
print("Transaction matrix shape:", transactions.shape)
transactions.head()


Transaction matrix shape: (6307, 116)


,Vegetarian's Delight,BBQ Chicken Pizza,Smoked Chicken Sandwich,Penne Alfredo,Pepsi,Mountain Dew,Seven up,Water 500ml,Cappuccino,Cream of Mushroom,...,Prawn-Lime Pizza,Coke,Americano Misto,Almond Cookie,Ceylon Supreme,Chicken Caesar Salad,Citrus Bliss,Kaffeehaus Special,Full English Breakfast,Light English Breakfast
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 4A.3 Original Total Sales Baseline

The original total sales are calculated before the discount simulation because the assignment asks whether the proposed package discount increases final revenue.

In [ ]:
df_sales = df.copy()
df_sales["TotalSales"] = 0.0

for dish, price in zip(dish_cols, price_cols):
    df_sales["TotalSales"] += df_sales[dish].fillna(0) * df_sales[price].fillna(0)

original_total_sales = df_sales["TotalSales"].sum()

print(f"Original Total Sales = {original_total_sales:,.2f}")


Original Total Sales = 5,111,310.00


In [ ]:
# transactions is (n_orders x n_items) with 0/1
# Convert each order into a set of items bought

transaction_sets = []
for i in range(len(transactions)):
    items = set(transactions.columns[transactions.iloc[i].values == 1])
    transaction_sets.append(items)

n = len(transaction_sets)
print("Total transactions:", n)
print("Example transaction:", list(transaction_sets[0])[:10])


Total transactions: 6307
Example transaction: ["Vegetarian's Delight"]


### 4A.4 Support Definition

Support is the fraction of total orders containing a given itemset.

In [ ]:
def support_count(itemset, transaction_sets):
    """Return how many transactions contain itemset."""
    cnt = 0
    for t in transaction_sets:
        if itemset.issubset(t):
            cnt += 1
    return cnt

def support(itemset, transaction_sets):
    return support_count(itemset, transaction_sets) / len(transaction_sets)


### 4A.5 Frequent 1-Itemsets

In [ ]:
min_support = 0.001  # or 0.002 based on assignment
min_count = int(np.ceil(min_support * n))

print("min_support:", min_support)
print("min_count (absolute):", min_count)

# Count each single item
item_counts = {}
for t in transaction_sets:
    for item in t:
        item_counts[item] = item_counts.get(item, 0) + 1

L1 = {frozenset([item]) for item, c in item_counts.items() if c >= min_count}
print("Frequent 1-itemsets:", len(L1))


min_support: 0.001
min_count (absolute): 7
Frequent 1-itemsets: 105


### 4A.6 Candidate Generation

In [ ]:
def generate_candidates(L_prev, k):
    """
    Generate candidate k-itemsets from frequent (k-1)-itemsets.
    Join step: combine sets that share first k-2 items (sorted).
    """
    L_prev_list = sorted([sorted(list(s)) for s in L_prev])
    candidates = set()

    for i in range(len(L_prev_list)):
        for j in range(i + 1, len(L_prev_list)):
            a = L_prev_list[i]
            b = L_prev_list[j]

            # if first k-2 items match, join
            if a[:k-2] == b[:k-2]:
                cand = frozenset(a) | frozenset(b)
                if len(cand) == k:
                    candidates.add(cand)
            else:
                break
    return candidates


### 4A.7 Apriori Pruning

In [ ]:
import itertools

def prune_candidates(Ck, L_prev, k):
    pruned = set()
    L_prev_set = set(L_prev)

    for cand in Ck:
        all_subsets_frequent = True
        for subset in itertools.combinations(cand, k-1):
            if frozenset(subset) not in L_prev_set:
                all_subsets_frequent = False
                break
        if all_subsets_frequent:
            pruned.add(cand)
    return pruned


### 4A.8 Support Counting and Frequent Itemset Selection

In [ ]:
def make_frequent(Ck, transaction_sets, min_count):
    counts = {}
    for cand in Ck:
        counts[cand] = support_count(set(cand), transaction_sets)
    Lk = {cand for cand, c in counts.items() if c >= min_count}
    return Lk, counts


### 4A.9 Run Apriori up to Four-Item Packages

In [ ]:
# Store all frequent itemsets and counts by size
frequent_by_k = {}
counts_by_k = {}

# L1
frequent_by_k[1] = L1

# Iteratively build L2, L3, L4
L_prev = L1
for k in [2, 3, 4]:
    print(f"\n=== k = {k} ===")
    Ck = generate_candidates(L_prev, k)
    print("Candidates before prune:", len(Ck))

    Ck = prune_candidates(Ck, L_prev, k)
    print("Candidates after prune:", len(Ck))

    Lk, counts = make_frequent(Ck, transaction_sets, min_count)
    frequent_by_k[k] = Lk
    counts_by_k[k] = counts

    print("Frequent itemsets:", len(Lk))
    L_prev = Lk

# Top 4-itemsets by support
L4 = frequent_by_k.get(4, set())
if len(L4) == 0:
    print("\nNo frequent 4-itemsets found.")
else:
    top4 = sorted(
        [(iset, counts_by_k[4][iset] / n) for iset in L4],
        key=lambda x: x[1],
        reverse=True
    )[:10]

    print("\nTop 10 frequent 4-itemsets:")
    for iset, sup in top4:
        print(list(iset), "support =", round(sup, 6))



=== k = 2 ===
Candidates before prune: 5460
Candidates after prune: 5460
Frequent itemsets: 592

=== k = 3 ===
Candidates before prune: 4788
Candidates after prune: 2865
Frequent itemsets: 304

=== k = 4 ===
Candidates before prune: 145
Candidates after prune: 104
Frequent itemsets: 11

Top 10 frequent 4-itemsets:
['Water 500ml', 'BBQ Chicken  Pizza', 'Pepsi', 'Mountain Dew'] support = 0.001903
['Water 500ml', 'French Fries', 'Pepsi', 'Chicken Cheese Balls'] support = 0.001586
['Water 500ml', 'French Fries', 'Pepsi', 'Mountain Dew'] support = 0.001586
['Water 500ml', 'Pepsi', 'Chicken Cheese Balls', 'Mountain Dew'] support = 0.001427
['Water 500ml', 'Mirinda', 'Pepsi', 'Chicken Supreme Pizza'] support = 0.001268
['Water 500ml', 'French Fries', 'BBQ Chicken  Pizza', 'Pepsi'] support = 0.00111
['Potato Wedges', 'Water 500ml', 'Pepsi', 'Mountain Dew'] support = 0.00111
['Water 500ml', 'Pepsi', 'Chicken Supreme Pizza', 'Mountain Dew'] support = 0.00111
['Chicken Wings', 'BBQ Chicken  Pizz

### 4A.10 Discount Simulation for the Selected Four-Item Package

In [ ]:
# Step 4: pick most popular 4-item-package (highest support)
top_package_set, top_package_support = top4[0]     # frozenset, float
package_items = sorted(list(top_package_set))      # list of 4 dish names

print("Most popular 4-item package:", package_items)
print("Support:", top_package_support, f"({top_package_support*100:.3f}%)")


Most popular 4-item package: ['BBQ Chicken  Pizza', 'Mountain Dew', 'Pepsi', 'Water 500ml']
Support: 0.0019026478515934676 (0.190%)


#### Orders Containing the Full Package

Only orders containing all four package items are counted as package orders.

In [ ]:
# Orders that contain ALL 4 items
package_mask = np.ones(len(transactions), dtype=bool)
for item in package_items:
    package_mask &= (transactions[item] == 1)

num_package_orders = int(package_mask.sum())
print("Orders containing the full package:", num_package_orders)
print("Percent of all orders:", num_package_orders / len(transactions) * 100)


Orders containing the full package: 12
Percent of all orders: 0.19026478515934675


#### Original Revenue from Package Items

In [ ]:
# Helper: get the matching price column for a dish column (same index)
dish_to_price = {dish_cols[i]: price_cols[i] for i in range(len(dish_cols))}

original_package_revenue = 0.0
item_revenue_breakdown = {}

for item in package_items:
    pcol = dish_to_price[item]
    rev = (df_sales.loc[package_mask, item].fillna(0) * df_sales.loc[package_mask, pcol].fillna(0)).sum()
    item_revenue_breakdown[item] = rev
    original_package_revenue += rev

print("Original package revenue breakdown:")
for k, v in item_revenue_breakdown.items():
    print(f"  {k}: {v:,.2f}")
print("Total original package revenue:", f"{original_package_revenue:,.2f}")


Original package revenue breakdown:
  BBQ Chicken  Pizza: 10,105.00
  Mountain Dew: 2,200.00
  Pepsi: 2,200.00
  Water 500ml: 1,240.00
Total original package revenue: 15,745.00


#### Revenue Change After 5% Discount and 5% Increased Package Sales

In [ ]:
discount_rate = 0.05
sales_increase = 0.05

new_package_revenue = original_package_revenue * (1 - discount_rate) * (1 + sales_increase)

package_revenue_change = new_package_revenue - original_package_revenue

print("Original package revenue:", f"{original_package_revenue:,.2f}")
print("New package revenue (0.95 * 1.05):", f"{new_package_revenue:,.2f}")
print("Change due to package discount+uplift:", f"{package_revenue_change:,.2f}")


Original package revenue: 15,745.00
New package revenue (0.95 * 1.05): 15,705.64
Change due to package discount+uplift: -39.36


#### Associated Item Uplift

For other items ordered together with the selected package, the notebook estimates additional uplift using conditional probability: p(item | package).

In [ ]:
associated_uplift_revenue = 0.0
assoc_details = []

package_order_count = package_mask.sum()

for dish in dish_cols:
    if dish in package_items:
        continue

    # p(dish | package) based on presence in package orders
    both = package_mask & (transactions[dish] == 1)
    prob = both.sum() / package_order_count if package_order_count > 0 else 0.0

    if prob > 0:
        pcol = dish_to_price[dish]
        # original revenue of this dish inside package-orders (original price)
        original_rev_in_package_orders = (
            df_sales.loc[package_mask, dish].fillna(0) * df_sales.loc[package_mask, pcol].fillna(0)
        ).sum()

        # 5% increased sales for that revenue portion
        uplift = original_rev_in_package_orders * sales_increase
        associated_uplift_revenue += uplift

        assoc_details.append((dish, prob, original_rev_in_package_orders, uplift))

# Show top 15 associated by probability
assoc_details.sort(key=lambda x: x[1], reverse=True)

print("Top associated items by p(item | package):")
for dish, prob, base_rev, uplift in assoc_details[:15]:
    print(f"{dish:35s}  p={prob:.3f}  base_rev={base_rev:,.2f}  uplift(5%)={uplift:,.2f}")

print("\nTotal uplift revenue from associated items:", f"{associated_uplift_revenue:,.2f}")


Top associated items by p(item | package):
Chicken Wings                        p=0.333  base_rev=1,450.00  uplift(5%)=72.50
Chicken Supreme Pizza                p=0.250  base_rev=2,525.00  uplift(5%)=126.25
Chicken Cheese Balls                 p=0.167  base_rev=350.00  uplift(5%)=17.50
Pasta Basta                          p=0.167  base_rev=640.00  uplift(5%)=32.00
Mirinda                              p=0.167  base_rev=100.00  uplift(5%)=5.00
Smoked Chicken Sandwich              p=0.083  base_rev=170.00  uplift(5%)=8.50
Seven up                             p=0.083  base_rev=50.00  uplift(5%)=2.50
Cream of Mushroom                    p=0.083  base_rev=210.00  uplift(5%)=10.50
Potato Wedges                        p=0.083  base_rev=480.00  uplift(5%)=24.00
French Fries                         p=0.083  base_rev=120.00  uplift(5%)=6.00
Chicken Cashew Nut Salad             p=0.083  base_rev=290.00  uplift(5%)=14.50
Thai Fried Rice                      p=0.083  base_rev=300.00  uplift(5%)=15.

### 4A.11 Final Sales Comparison

In [ ]:
total_revenue_change = package_revenue_change + associated_uplift_revenue
new_total_sales = original_total_sales + total_revenue_change
percentage_change = (total_revenue_change / original_total_sales) * 100

print("\n" + "="*60)
print("FINAL SALES COMPARISON (Assignment 4A)")
print("="*60)
print("Original total sales:", f"{original_total_sales:,.2f}")
print("New total sales:", f"{new_total_sales:,.2f}")
print("Revenue change:", f"{total_revenue_change:,.2f}")
print("Percent change:", f"{percentage_change:+.3f}%")

if percentage_change > 0:
    print("Conclusion: Discount strategy INCREASED total sales.")
else:
    print("Conclusion: Discount strategy DECREASED total sales.")
print("="*60)



FINAL SALES COMPARISON (Assignment 4A)
Original total sales: 5,111,310.00
New total sales: 5,111,661.89
Revenue change: 351.89
Percent change: +0.007%
Conclusion: Discount strategy INCREASED total sales.


## Part 4B: Projection, Covariance, and PCA-Style Analysis

In [ ]:
import os
import pandas as pd
import numpy as np



In [ ]:
# Download the assigned numeric dataset for Part 4B.
# The notebook uses the diabetes dataset because it has numeric dimensions and a class label.

import kagglehub

path = kagglehub.dataset_download("mathchi/diabetes-data-set")
print("Path to dataset files:", path)


In [ ]:
import os
import pandas as pd

# List files inside the downloaded dataset folder
files = os.listdir(path)
print(files)

# Load the CSV file
csv_file = [f for f in files if f.endswith(".csv")][0]
csv_path = os.path.join(path, csv_file)

df_4b = pd.read_csv(csv_path)

print("Shape of raw dataset:", df_4b.shape)
df_4b.head()


['diabetes.csv']
Shape of raw dataset: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
import numpy as np

# Copy raw data for cleaning
df_4b_clean = df_4b.copy()

print("Initial shape:", df_4b_clean.shape)

# 1. Replace blank strings with NaN (defensive step)
df_4b_clean.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Diabetes-specific invalid zero handling
invalid_zero_cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for col in invalid_zero_cols:
    if col in df_4b_clean.columns:
        df_4b_clean[col] = df_4b_clean[col].replace(0, np.nan)

# 3. Drop rows with missing values
df_4b_clean = df_4b_clean.dropna().reset_index(drop=True)

print("Shape after cleaning:", df_4b_clean.shape)

# 4. Ensure numeric types (except label)
label_col = df_4b_clean.columns[-1]

for c in df_4b_clean.columns[:-1]:
    df_4b_clean[c] = pd.to_numeric(df_4b_clean[c], errors="coerce")

# Final sanity check
print("\nRemaining missing values:")
print(df_4b_clean.isna().sum())

df_4b_clean.head()


Initial shape: (768, 9)
Shape after cleaning: (392, 9)

Remaining missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
1,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
2,3,78.0,50.0,32.0,88.0,31.0,0.248,26,1
3,2,197.0,70.0,45.0,543.0,30.5,0.158,53,1
4,1,189.0,60.0,23.0,846.0,30.1,0.398,59,1


### 4B.1 Original 2D Interactive Plot

In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Separate features and label ---
label_col = df_4b_clean.columns[-1]     # usually "Outcome" for this dataset
feature_cols = df_4b_clean.columns[:-1]

X = df_4b_clean[feature_cols].to_numpy()
y = df_4b_clean[label_col].to_numpy()

classes = np.unique(y)
print("Label column:", label_col)
print("Classes found:", classes, "| Number of classes:", len(classes))

# --- First two dimensions ---
x1 = X[:, 0]
x2 = X[:, 1]
x1_name = feature_cols[0]
x2_name = feature_cols[1]

# --- Marker shapes (assignment asks different shapes) ---
symbols = ["circle", "square", "diamond", "x", "triangle-up"]  # enough for more classes
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=x1[mask],
        y=x2[mask],
        mode="markers",
        name=f"Class {cls}",
        marker=dict(size=9, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(a)(i) Original Data (First Two Dimensions)",
    xaxis_title=x1_name,
    yaxis_title=x2_name
)

fig.show()


Label column: Outcome
Classes found: [0 1] | Number of classes: 2


In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Separate features and label ---
label_col = df_4b_clean.columns[-1]
feature_cols = df_4b_clean.columns[:-1]

X = df_4b_clean[feature_cols].to_numpy()
y = df_4b_clean[label_col].to_numpy()
classes = np.unique(y)

# --- Take first two dimensions ---
X2 = X[:, :2]   # shape: (n_samples, 2)

# --- Centering: subtract mean of each dimension ---
mean_2d = X2.mean(axis=0)
X2_centered = X2 - mean_2d

print("Mean of original first two dimensions:", mean_2d)
print("Mean after centering (should be ~0):", X2_centered.mean(axis=0))

# --- Plot centered data (Plotly 2D, interactive) ---
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=X2_centered[mask, 0],
        y=X2_centered[mask, 1],
        mode="markers",
        name=f"Class {cls}",
        marker=dict(size=9, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(a)(ii) Centered Data (First Two Dimensions)",
    xaxis_title=f"Centered {feature_cols[0]}",
    yaxis_title=f"Centered {feature_cols[1]}"
)

fig.show()


Mean of original first two dimensions: [  3.30102041 122.62755102]
Mean after centering (should be ~0): [2.53765263e-16 5.80034886e-16]


In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Reuse centered data from 4B(a)(ii) ---
# X2_centered: (n_samples, 2)
# y: class labels
# feature_cols: feature names

# Direction vector for the line x1 = -x2
v = np.array([1.0, -1.0])
v = v / np.linalg.norm(v)  # unit direction

# Parameter range for drawing the line
t = np.linspace(-5, 5, 300)
line_points = np.outer(t, v)  # shape (300, 2)

# --- Plot centered data + the line ---
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

# Centered data points
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=X2_centered[mask, 0],
        y=X2_centered[mask, 1],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=9, symbol=class_to_symbol[cls])
    ))

# The line x1 = -x2
fig.add_trace(go.Scatter(
    x=line_points[:, 0],
    y=line_points[:, 1],
    mode="lines",
    name="Line: x1 = -x2",
    line=dict(width=3)
))

fig.update_layout(
    title="4B(a)(iii) Centered Data with Line x1 = -x2",
    xaxis_title=f"Centered {feature_cols[0]}",
    yaxis_title=f"Centered {feature_cols[1]}"
)

fig.show()


### 4B.4 Projection onto the Line x1 = -x2

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Direction vector for the line x1 = -x2  (span{[1,-1]})
u = np.array([1.0, -1.0])
u = u / np.linalg.norm(u)  # unit vector

# Projection of each point x onto the line: proj(x) = (x·u)u
scalars = X2_centered @ u                 # (n,)
X2_proj = np.outer(scalars, u)            # (n,2)

# Line points for plotting
t = np.linspace(-5, 5, 300)
line_points = np.outer(t, u)

# Plot centered data + line + projections
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

# Centered original points
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=X2_centered[mask, 0],
        y=X2_centered[mask, 1],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=9, symbol=class_to_symbol[cls])
    ))

# The line x1 = -x2
fig.add_trace(go.Scatter(
    x=line_points[:, 0],
    y=line_points[:, 1],
    mode="lines",
    name="Line: x1 = -x2",
    line=dict(width=3)
))

# Projected points (same symbol, smaller size)
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=X2_proj[mask, 0],
        y=X2_proj[mask, 1],
        mode="markers",
        name=f"Projected Class {cls}",
        marker=dict(size=5, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(a)(iv) Projection onto Line x1 = -x2",
    xaxis_title=f"Centered {feature_cols[0]}",
    yaxis_title=f"Centered {feature_cols[1]}"
)

fig.show()


### 4B.5 Original 3D Interactive Plot

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Separate features and label
label_col = df_4b_clean.columns[-1]
feature_cols = df_4b_clean.columns[:-1]

X = df_4b_clean[feature_cols].to_numpy()
y = df_4b_clean[label_col].to_numpy()
classes = np.unique(y)

# First three dimensions
X3 = X[:, :3]
x_name, y_name, z_name = feature_cols[0], feature_cols[1], feature_cols[2]

# Marker symbols per class
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=X3[mask, 0],
        y=X3[mask, 1],
        z=X3[mask, 2],
        mode="markers",
        name=f"Class {cls}",
        marker=dict(size=4, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(b)(i) Original Data (First Three Dimensions)",
    scene=dict(
        xaxis_title=x_name,
        yaxis_title=y_name,
        zaxis_title=z_name
    )
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Separate features and label
label_col = df_4b_clean.columns[-1]
feature_cols = df_4b_clean.columns[:-1]

X = df_4b_clean[feature_cols].to_numpy()
y = df_4b_clean[label_col].to_numpy()
classes = np.unique(y)

# First three dimensions
X3 = X[:, :3]

# Centering: subtract mean vector
mean_3d = X3.mean(axis=0)
D3 = X3 - mean_3d   # centered 3D data (we'll reuse this later)

print("Mean of original first three dims:", mean_3d)
print("Mean after centering (should be ~0):", D3.mean(axis=0))

x_name, y_name, z_name = feature_cols[0], feature_cols[1], feature_cols[2]

# Marker symbols per class
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

# Plot centered 3D data
fig = go.Figure()

for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3[mask, 0],
        y=D3[mask, 1],
        z=D3[mask, 2],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=4, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(b)(ii) Centered Data (First Three Dimensions)",
    scene=dict(
        xaxis_title=f"Centered {x_name}",
        yaxis_title=f"Centered {y_name}",
        zaxis_title=f"Centered {z_name}"
    )
)

fig.show()


Mean of original first three dims: [  3.30102041 122.62755102  70.66326531]
Mean after centering (should be ~0): [ 2.53765263e-16  5.80034886e-16 -5.51033142e-15]


In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Given spanning vectors ---
v1 = np.array([1.0, -2.0, 1.0])
v2 = np.array([2.0,  1.0, 0.0])

# --- Create a grid of points on the plane: s*v1 + t*v2 ---
s = np.linspace(-5, 5, 25)
t = np.linspace(-5, 5, 25)
S, T = np.meshgrid(s, t)

plane = (S[..., None] * v1) + (T[..., None] * v2)  # shape (25,25,3)
px, py, pz = plane[..., 0], plane[..., 1], plane[..., 2]

# --- Plot centered data + plane ---
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

# Centered data points
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3[mask, 0],
        y=D3[mask, 1],
        z=D3[mask, 2],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=4, symbol=class_to_symbol[cls])
    ))

# Plane surface
fig.add_trace(go.Surface(
    x=px, y=py, z=pz,
    opacity=0.45,
    showscale=False,
    name="Plane: span(v1, v2)"
))

fig.update_layout(
    title="4B(b)(iii) Centered Data with Plane span(v1, v2)",
    scene=dict(
        xaxis_title=f"Centered {feature_cols[0]}",
        yaxis_title=f"Centered {feature_cols[1]}",
        zaxis_title=f"Centered {feature_cols[2]}"
    )
)

fig.show()


### 4B.8 Projection onto the Given Plane

In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Given spanning vectors ---
v1 = np.array([1.0, -2.0, 1.0])
v2 = np.array([2.0,  1.0, 0.0])

# --- Orthonormal basis for the plane using QR decomposition ---
V = np.column_stack([v1, v2])   # shape (3,2)
Q, _ = np.linalg.qr(V)          # Q is (3,2), orthonormal basis of span(v1,v2)

# --- Projection matrix onto the plane ---
P_plane = Q @ Q.T               # shape (3,3)

# --- Project centered data points ---
D3_proj = (P_plane @ D3.T).T    # shape (n,3)

# --- Create the plane surface again for visualization ---
s = np.linspace(-5, 5, 25)
t = np.linspace(-5, 5, 25)
S, T = np.meshgrid(s, t)
plane = (S[..., None] * v1) + (T[..., None] * v2)

px, py, pz = plane[..., 0], plane[..., 1], plane[..., 2]

# --- Plot centered data + plane + projected points ---
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

# Centered original points
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3[mask, 0], y=D3[mask, 1], z=D3[mask, 2],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=4, symbol=class_to_symbol[cls])
    ))

# Plane surface
fig.add_trace(go.Surface(
    x=px, y=py, z=pz,
    opacity=0.45,
    showscale=False,
    name="Plane: span(v1, v2)"
))

# Projected points (smaller markers)
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3_proj[mask, 0], y=D3_proj[mask, 1], z=D3_proj[mask, 2],
        mode="markers",
        name=f"Projected Class {cls}",
        marker=dict(size=2.5, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(b)(iv) Projection onto Plane span(v1, v2)",
    scene=dict(
        xaxis_title=f"Centered {feature_cols[0]}",
        yaxis_title=f"Centered {feature_cols[1]}",
        zaxis_title=f"Centered {feature_cols[2]}"
    )
)

fig.show()


## 4B.9 Numeric Data Analysis

### Covariance Matrix from Centered Data

In [ ]:
import numpy as np

# D3 is centered 3D data: shape (n, 3)
n = D3.shape[0]

# Covariance matrix using inner product form
C = (D3.T @ D3) / n

print("n (number of samples):", n)
print("Covariance matrix C shape:", C.shape)
print("Covariance matrix C:\n", C)


n (number of samples): 392
Covariance matrix C shape: (3, 3)
Covariance matrix C:
 [[ 10.28693773  19.60191066   8.54013953]
 [ 19.60191066 949.95822053  80.78784881]
 [  8.54013953  80.78784881 155.75395668]]


### Eigenvalues, Eigenvectors, and Orthonormality Check

In [ ]:
import numpy as np

# Eigen decomposition for symmetric matrices
eigvals, eigvecs = np.linalg.eigh(C)  # eigvals ascending by default

# Sort in descending order
idx = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[idx]
eigvecs_sorted = eigvecs[:, idx]

print("Eigenvalues (descending):", eigvals_sorted)
print("\nEigenvectors matrix shape:", eigvecs_sorted.shape)  # should be (3,3)

# Sanity check: eigenvectors should be orthonormal -> V^T V = I
I_check = eigvecs_sorted.T @ eigvecs_sorted
print("\nOrthonormality check (V^T V):\n", I_check)


Eigenvalues (descending): [958.52991598 147.92836195   9.54083702]

Eigenvectors matrix shape: (3, 3)

Orthonormality check (V^T V):
 [[ 1.00000000e+00  1.32080615e-17 -1.42726768e-17]
 [ 1.32080615e-17  1.00000000e+00  2.14517899e-18]
 [-1.42726768e-17  2.14517899e-18  1.00000000e+00]]


### Principal Subspace Matrix U

In [ ]:
import numpy as np

# U contains the top 2 eigenvectors (3x2)
U = eigvecs_sorted[:, :2]

print("U shape:", U.shape)  # should be (3,2)
print("U:\n", U)

# Sanity check: U should have orthonormal columns -> U^T U = I_2
print("\nU^T U:\n", U.T @ U)


U shape: (3, 2)
U:
 [[-0.02146635  0.04723806]
 [-0.99472233 -0.10125304]
 [-0.10033285  0.99373859]]

U^T U:
 [[1.00000000e+00 1.32080615e-17]
 [1.32080615e-17 1.00000000e+00]]


### Projection Using x' = U Uᵀx

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Projection matrix onto the subspace spanned by U
P_pca = U @ U.T            # shape (3,3)

# Project centered 3D data
D3_proj_pca = (P_pca @ D3.T).T   # shape (n,3)

print("Projected data shape:", D3_proj_pca.shape)

# --- Plot centered data + PCA plane + projected points ---
# Create a plane for visualization: span(u1, u2)
u1, u2 = U[:, 0], U[:, 1]
s = np.linspace(-5, 5, 25)
t = np.linspace(-5, 5, 25)
S, T = np.meshgrid(s, t)
pca_plane = (S[..., None] * u1) + (T[..., None] * u2)

px, py, pz = pca_plane[..., 0], pca_plane[..., 1], pca_plane[..., 2]

symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

# Centered original points
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3[mask, 0], y=D3[mask, 1], z=D3[mask, 2],
        mode="markers",
        name=f"Centered Class {cls}",
        marker=dict(size=4, symbol=class_to_symbol[cls])
    ))

# PCA plane
fig.add_trace(go.Surface(
    x=px, y=py, z=pz,
    opacity=0.45,
    showscale=False,
    name="Plane: span(u1, u2)"
))

# Projected points (smaller markers)
for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter3d(
        x=D3_proj_pca[mask, 0],
        y=D3_proj_pca[mask, 1],
        z=D3_proj_pca[mask, 2],
        mode="markers",
        name=f"Projected Class {cls}",
        marker=dict(size=2.5, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(c)(iv) Projection using x' = U U^T x",
    scene=dict(
        xaxis_title=f"Centered {feature_cols[0]}",
        yaxis_title=f"Centered {feature_cols[1]}",
        zaxis_title=f"Centered {feature_cols[2]}"
    )
)

fig.show()


Projected data shape: (392, 3)


### 2D Coordinates in the U-Basis: [x']ᵤ = Uᵀx

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 2D coordinates in the U-basis
Z = (U.T @ D3.T).T   # shape (n,2)

print("Z shape (new 2D coordinates):", Z.shape)
print("First 5 rows of Z:\n", Z[:5])

# Plot 2D coordinates with class-based marker shapes
symbols = ["circle", "square", "diamond", "x", "triangle-up"]
classes = np.unique(y)
class_to_symbol = {cls: symbols[i] for i, cls in enumerate(classes)}

fig = go.Figure()

for cls in classes:
    mask = (y == cls)
    fig.add_trace(go.Scatter(
        x=Z[mask, 0],
        y=Z[mask, 1],
        mode="markers",
        name=f"Class {cls}",
        marker=dict(size=9, symbol=class_to_symbol[cls])
    ))

fig.update_layout(
    title="4B(c)(v) 2D Coordinates: [x']_U = U^T x",
    xaxis_title="Coordinate along u1",
    yaxis_title="Coordinate along u2"
)

fig.show()


Z shape (new 2D coordinates): (392, 2)
First 5 rows of Z:
 [[ 33.96734914  -1.33787075]
 [-11.14920217 -32.08245798]
 [ 46.47168773 -16.02942869]
 [-73.8854603   -8.25100636]
 [-64.90288677 -17.42560603]]
